# Data Collection Workflow

### Required libraries

In [ ]:
!python.exe -m pip install -U pip
!pip install selenium beautifulsoup4 pandas undetected-chromedriver

In [11]:
import requests, re, json
from bs4 import BeautifulSoup
import pandas as pd

### Retrieval data

In [ ]:
# List of boxer names to scrape (you can add more names or load from a file)
boxers = {
    "Oleksandr Usyk",
    "Tyson Fury",
    "Daniel Dubois",
    "Joseph Parker",
    "Agit Kabayel",
    "Fabio Wardley",
    "Martin Bakole",
    "Derek Chisora",
    "Michael Hunter",
    "Frank Sanchez",
    "Moses Itauma",
    "Jared Anderson",
    "Kubrat Pulev",
    "Jarrell Miller",
    "Dillian Whyte",
    "Andy Ruiz Jr.",
    "Murat Gassiev",
    "Justis Huni",
    "Joe Joyce",
    "Jai Opetaia",
    "Gilberto Ramirez",
    "Chris Billam‑Smith",
    "Badou Jack",
    "Norair Mikaeljan",
    "Evgeny Romanov",
    "Leon Harth",
    "Andrew Tabiti",
    "Kevin Lerena",
    "Dmitry Bivol",
    "Artur Beterbiev",
    "David Benavidez",
    "David Morrell Jr.",
    "Callum Smith",
    "Joshua Buatsi",
    "Efe Ajagba",
    "Canelo Álvarez",
    "Christian Mbilli",
    "Diego Pacheco",
    "Edgar Berlanga",
    "Jermall Charlo",
    "Caleb Plant",
    "Janibek Alimkhanuly",
    "Carlos Adames",
    "Hamzah Sheeraz",
    "Erislandy Lara",
    "Chris Eubank Jr.",
    "Terence Crawford",
    "Vergil Ortiz Jr.",
    "Sebastian Fundora",
    "Bakhram Murtazaliev",
    "Israil Madrimov",
    "Tim Tszyu",
    "Jaron Ennis",
    "Brian Norman Jr.",
    "Mario Barrios",
    "Eimantas Stanionis",
    "Devin Haney",
    "Teofimo Lopez",
    "Richardson Hitchins",
    "Alberto Puello",
    "Arnold Barboza Jr.",
    "Gary Antuanne Russell",
    "Liam Paro",
    "Subriel Matías",
    "Sandor Martín",
    "Gervonta Davis",
    "Shakur Stevenson",
    "Keyshawn Davis",
    "William Zepeda",
    "Lamont Roach",
    "Andy Cruz",
    "Emanuel Navarrete",
    "Anthony Cacace",
    "O’Shaiquie Foster",
    "Jesse Rodriguez",
    "Fernando Martinez",
    "Kazuto Ioka",
    "Phumelele Cafu",
    "Carlos Cuadras",
    "Kosei Tanaka",
    "Naoya Inoue",
    "Marlon Tapales",
    "Murodjon Akhmadaliev",
    "Sam Goodman",
    "Luis Nery",
    "Alan Picasso Romero",
    "Junto Nakatani",
    "Ryosuke Nishida",
    "Kenshiro Teraji",
    "Seigo Yuri Akui",
    "Francisco Rodriguez Jr.",
    "Ricardo Sandoval",
    "Masamichi Yabuki",
    "Rene Santiago",
    "Jonathan Gonzalez",
    "Elwin Soto",
    "Petchmanee CP Freshmart",
    "Shokichi Iwata",
    "Oscar Collazo",
    "Pedro Taduran"
}


# Function to scrape one boxer's data from Wikipedia
def scrape_boxer_info(name):
    # Construct Wikipedia URL (replace spaces with underscores)
    url = "https://en.wikipedia.org/wiki/" + name.replace(' ', '_')
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    if resp.status_code != 200:
        print(f"Failed to retrieve page for {name} (status {resp.status_code})")
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Find the infobox table in the page
    infobox = soup.find("table", {"class": "infobox"})
    if infobox is None:
        print(f"No infobox found for {name} – skipping.")
        return None
    
    data = {"Name": name}
    # Go through each table row in the infobox
    for row in infobox.find_all("tr"):
        header = row.find("th")
        value = row.find("td")
        if not header or not value:
            continue  # skip rows that are not "header: value" pairs (e.g. section headers)
        field = header.get_text(strip=True)
        val_text = value.get_text(" ", strip=True)  # get text inside td
        
        # Extract relevant fields
        if field.startswith("Weight"):  # "Weight(s)"
            # If multiple weight classes are listed as a <ul> list, join them
            # Check for list items in the value
            weight_classes = [li.get_text(strip=True) for li in value.find_all("li")]
            if weight_classes:
                data["Weight"] = ", ".join(weight_classes)
            else:
                data["Weight"] = val_text
        elif field == "Height":
            data["Height"] = val_text
        elif field == "Reach":
            data["Reach"] = val_text
        elif field == "Stance" or field == "Style":
            data["Stance"] = val_text
        elif field == "Total fights":
            data["Total_fights"] = val_text
        elif field == "Wins":
            data["Wins"] = val_text
        elif field == "Losses":
            data["Losses"] = val_text
        elif field == "Draws":
            data["Draws"] = val_text
        elif field == "No contests": 
            data["No_contests"] = val_text
        elif field == "Born":
            age_match = re.search(r'\(age\s+(\d+)\)', val_text)
            if age_match:
                data["Age"] = age_match.group(1)
            else:
                data["Age"] = None  # age not found (maybe the person is deceased or info not available)
        # (We skip other fields like nationality, etc., not requested)
    
    # If age wasn't in Born (e.g., deceased boxers won't have an age there), check for Died field
    if "Age" not in data:
        died_field = infobox.find("th", string="Died")
        if died_field:
            died_text = died_field.find_next("td").get_text(" ", strip=True)
            age_match = re.search(r'\(aged\s+(\d+)\)', died_text)
            if age_match:
                data["Age"] = age_match.group(1)  # age at death
    
    return data


# Scrape data for all boxers in the list
dataset = []
for name in boxers:
    info = scrape_boxer_info(name)
    if info:
        dataset.append(info)


# Display the collected data (for example, print each boxer's info)
for boxer in dataset:
    print(boxer)

Failed to retrieve page for David Morrell Jr. (status 404)
Failed to retrieve page for Petchmanee CP Freshmart (status 404)
Failed to retrieve page for Petchmanee CP Freshmart (status 404)
No infobox found for Luis Nery – skipping.
No infobox found for Luis Nery – skipping.
Failed to retrieve page for Chris Billam‑Smith (status 404)
Failed to retrieve page for Chris Billam‑Smith (status 404)
Failed to retrieve page for O’Shaiquie Foster (status 404)
Failed to retrieve page for O’Shaiquie Foster (status 404)
No infobox found for Jonathan Gonzalez – skipping.
No infobox found for Jonathan Gonzalez – skipping.
Failed to retrieve page for Lamont Roach (status 404)
Failed to retrieve page for Lamont Roach (status 404)
Failed to retrieve page for Francisco Rodriguez Jr. (status 404)
Failed to retrieve page for Francisco Rodriguez Jr. (status 404)
No infobox found for Michael Hunter – skipping.
No infobox found for Michael Hunter – skipping.
Failed to retrieve page for Rene Santiago (status 4

### Save data

In [ ]:
dataset = pd.DataFrame(dataset)

# json.dump(
#     dataset.to_dict(orient='records'),
#     open('../data/boxers_data.json', 'w'),
#     indent=4
#     )

dataset.to_csv('../data/boxers_data.csv', index=False)